In [27]:
import json

In [28]:
with open(r"C:\Users\WIZBOX\Desktop\Projects\ReceiptRadar\data\raw\biedronka\json\paragon_2604265011012375.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [29]:
data['header'][2]

{'headerData': {'tin': '7791011327',
  'docNumber': 1017683,
  'date': '2026-04-26T17:33:02.000Z',
  'CPS': 56}}

In [30]:
header_data = next(
    h["headerData"] for h in data["header"] if "headerData" in h
)

In [31]:
receipt = {
    "receipt_id": data["IDZ"],
    "date": header_data["date"],
    "store": "biedronka"
}

In [32]:
print(receipt)

{'receipt_id': 'c=98936b53d199457493eaf72dd3d3b522|g={25abca98-c9cb-4e21-bf84-a72c0d0cbaf0}|s=2375|p=1|t=5011', 'date': '2026-04-26T17:33:02.000Z', 'store': 'biedronka'}


In [33]:
def clean_item(sl):
    return {
        "name": sl["name"].strip().rsplit(" ", 1)[0],
        "quantity": float(sl["quantity"].replace(",", ".")),
        "unit_price": sl["price"] / 100,
        "total_price": sl["total"] / 100
    }


items = []
current_item = None

for entry in data["body"]:
    if "sellLine" in entry:
        current_item = clean_item(entry["sellLine"])
        current_item["discount"] = 0
        items.append(current_item)

    elif "discountLine" in entry and current_item is not None:
        current_item["discount"] += entry["discountLine"]["value"] / 100

for item in items:
    item["final_price"] = item["total_price"] - item["discount"]
    item["receipt_id"] = receipt["receipt_id"]

In [34]:
import pandas as pd

df = pd.DataFrame(items)
df.head(10)

,name,quantity,unit_price,total_price,discount,final_price,receipt_id
0,Ręcznik Milla X2,2.000,4.69,9.38,1.40,7.98,c=98936b53d199457493eaf72dd3d3b522|g={25abca98...
1,SerekKlasDelik200g,2.000,4.69,9.38,2.00,7.38,c=98936b53d199457493eaf72dd3d3b522|g={25abca98...
2,JajaGoBio 9szt,2.000,14.39,28.78,0.00,28.78,c=98936b53d199457493eaf72dd3d3b522|g={25abca98...
3,ParowkiGłodniaki200g,1.000,5.49,5.49,0.00,5.49,c=98936b53d199457493eaf72dd3d3b522|g={25abca98...
4,Pom Paprycz Luz,0.494,29.99,14.82,7.41,7.41,c=98936b53d199457493eaf72dd3d3b522|g={25abca98...
5,Jog Naturalny 400g,1.000,1.79,1.79,0.00,1.79,c=98936b53d199457493eaf72dd3d3b522|g={25abca98...
6,MąkaOrkisPełnoz1kg,1.000,5.79,5.79,0.00,5.79,c=98936b53d199457493eaf72dd3d3b522|g={25abca98...
7,CiecNasSpi540 400g,1.000,6.49,6.49,0.00,6.49,c=98936b53d199457493eaf72dd3d3b522|g={25abca98...
8,Banan Luz,1.564,6.99,10.93,0.00,10.93,c=98936b53d199457493eaf72dd3d3b522|g={25abca98...
9,KefirProtGoActi 420g,4.000,4.49,17.96,0.00,17.96,c=98936b53d199457493eaf72dd3d3b522|g={25abca98...
